In [1]:
import pandas as pd

# Fuente A (CSV)
ventas = pd.read_csv("../data/raw/sales.csv")

# Fuente B (Excel)
hojas = pd.read_excel("../data/raw/maestro_tiendas_productos.xlsx", sheet_name=None)
productos = hojas["productos"]
tiendas = hojas["tiendas"]

# Limpieza de tipos
ventas["Date"] = pd.to_datetime(ventas["Date"])
tiendas["Store_Open_Date"] = pd.to_datetime(tiendas["Store_Open_Date"])

for col in ["Product_Cost", "Product_Price"]:
    productos[col] = (
        productos[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.strip()
        .astype(float)
    )

print(ventas.shape, productos.shape, tiendas.shape)
print(ventas.dtypes)
print(productos.dtypes)
print(tiendas.dtypes)

(829262, 5) (35, 5) (50, 5)
Sale_ID                int64
Date          datetime64[us]
Store_ID               int64
Product_ID             int64
Units                  int64
dtype: object
Product_ID            int64
Product_Name            str
Product_Category        str
Product_Cost        float64
Product_Price       float64
dtype: object
Store_ID                    int64
Store_Name                    str
Store_City                    str
Store_Location                str
Store_Open_Date    datetime64[us]
dtype: object


In [2]:
df = ventas.merge(productos, on="Product_ID", how="left")
df = df.merge(tiendas, on="Store_ID", how="left")

print("Filas y columnas:", df.shape)
print("Filas antes de unir:", len(ventas))
print("Valores vacíos tras unir:")
print(df.isna().sum())
print(df.head())

Filas y columnas: (829262, 13)
Filas antes de unir: 829262
Valores vacíos tras unir:
Sale_ID             0
Date                0
Store_ID            0
Product_ID          0
Units               0
Product_Name        0
Product_Category    0
Product_Cost        0
Product_Price       0
Store_Name          0
Store_City          0
Store_Location      0
Store_Open_Date     0
dtype: int64
   Sale_ID       Date  Store_ID  Product_ID  Units      Product_Name  \
0        1 2022-01-01        24           4      1  Chutes & Ladders   
1        2 2022-01-01        28           1      1     Action Figure   
2        3 2022-01-01         6           8      1     Deck Of Cards   
3        4 2022-01-01        48           7      1          Dart Gun   
4        5 2022-01-01        44          18      1       Lego Bricks   

    Product_Category  Product_Cost  Product_Price  \
0              Games          9.99          12.99   
1               Toys          9.99          15.99   
2              Games    

In [3]:
# Dinero
df["Total_Revenue"] = df["Units"] * df["Product_Price"]
df["Total_Cost"] = df["Units"] * df["Product_Cost"]
df["Profit"] = df["Total_Revenue"] - df["Total_Cost"]
df["Margin_Pct"] = (df["Profit"] / df["Total_Revenue"] * 100).round(2)

# Tiempo
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["Quarter"] = df["Date"].dt.quarter
df["Weekday"] = df["Date"].dt.day_name()
df["Is_Weekend"] = df["Date"].dt.dayofweek >= 5

# Tienda
df["Store_Age_Years"] = ((df["Date"] - df["Store_Open_Date"]).dt.days / 365.25).round(1)

print("Filas y columnas:", df.shape)
print(df[["Units", "Total_Revenue", "Total_Cost", "Profit", "Margin_Pct",
          "Year", "Month", "Weekday", "Is_Weekend", "Store_Age_Years"]].head())
print(df[["Total_Revenue", "Profit", "Margin_Pct", "Store_Age_Years"]].describe())

Filas y columnas: (829262, 24)
   Units  Total_Revenue  Total_Cost  Profit  Margin_Pct  Year  Month  \
0      1          12.99        9.99     3.0       23.09  2022      1   
1      1          15.99        9.99     6.0       37.52  2022      1   
2      1           6.99        3.99     3.0       42.92  2022      1   
3      1          15.99       11.99     4.0       25.02  2022      1   
4      1          39.99       34.99     5.0       12.50  2022      1   

    Weekday  Is_Weekend  Store_Age_Years  
0  Saturday        True             11.4  
1  Saturday        True             10.8  
2  Saturday        True             18.1  
3  Saturday        True              5.8  
4  Saturday        True              7.0  
       Total_Revenue         Profit     Margin_Pct  Store_Age_Years
count  829262.000000  829262.000000  829262.000000    829262.000000
mean       17.418587       4.840483      31.227174        13.301122
std        15.522069       4.476390      15.442033         5.374428
min   

In [5]:
df.to_csv("../data/processed/dataset_final.csv", index=False)
df.to_csv("../data/processed/dataset_final.csv.gz", index=False)

comprobacion = pd.read_csv("../data/processed/dataset_final.csv.gz", parse_dates=["Date", "Store_Open_Date"])
print(comprobacion.shape)

(829262, 24)


In [7]:
# Unión
df = ventas.merge(productos, on="Product_ID", how="left")
df = df.merge(tiendas, on="Store_ID", how="left")

# Dinero
df["Total_Revenue"] = df["Units"] * df["Product_Price"]
df["Total_Cost"] = df["Units"] * df["Product_Cost"]
df["Profit"] = df["Total_Revenue"] - df["Total_Cost"]
df["Margin_Pct"] = (df["Profit"] / df["Total_Revenue"] * 100).round(2)

# Tiempo
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.month_name()
df["Quarter"] = df["Date"].dt.quarter
df["Weekday"] = df["Date"].dt.day_name()
df["Is_Weekend"] = df["Date"].dt.dayofweek >= 5

# Tienda
df["Store_Age_Years"] = ((df["Date"] - df["Store_Open_Date"]).dt.days / 365.25).round(1)

# Guardar
df.to_csv("../data/processed/dataset_final.csv", index=False)
df.to_csv("../data/processed/dataset_final.csv.gz", index=False)

print(df.shape)
print("Cuidad de Mexico aún existe:", (df["Store_City"] == "Cuidad de Mexico").any())

(829262, 24)
Cuidad de Mexico aún existe: False
